# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T2 — Declare the grain

### Part A — the PakFreight grain statement

> **One row of FactShipment represents one shipment leg dispatched by PakFreight.**

No "and" — one thing. *Why "leg" and not "shipment"?* MR-1009 (Gwadar → Karachi) and RD-508
(Quetta → Karachi) are single-mode today, but the day PakFreight sells multi-modal freight
(marine-then-road for one consignment), "one row per shipment" becomes ambiguous — and the grain
declared today is the one the CFO's 2027 reports will be built on. "Leg" survives that future.

### Part B — four new processes

| # | Process | Grain (one row per …) | Dimensions | Measures | Fact type |
|---|---|---|---|---|---|
| 1 | Fuel purchases at depots | fuel purchase transaction | Date, Depot, Vehicle, Fuel Type | Litres, Cost PKR | **Transaction** — one row per event |
| 2 | Driver duty hours / rest compliance | driver per calendar day | Date, Driver, Vehicle | Duty hours, Rest hours | **Periodic snapshot** — regulators need a daily photo; rolling compliance can't come from transactions |
| 3 | Customer complaints | complaint | Date, Customer, Shipment (DD), Route | Complaints (=1), Days late, Refund PKR | **Transaction** — but one complaint can span shipments, needing a **bridge table** (not a dimension) |
| 4 | Daily warehouse stock, Lahore | commodity per warehouse per day | Date, Commodity, Warehouse | Closing stock KG | **Periodic snapshot** — nothing "happens" on a quiet day, yet the level must exist |

The two non-transaction tables: **2 and 4** (periodic snapshots). The non-dimension: **3's bridge table**.

### Part C — the grain that went wrong

The analyst broke the **grain rule**: the table promised one row per shipment leg, and CustomerName
doesn't fit that grain — duplicating the row double-counts the weight. He should have built a
shipment↔customer **bridge** (or billed at the customer×shipment grain). T7 check 4 (legs = 34)
and check 2 (578,650 kg) catch it the instant the row doubles.

In [1]:
# Grain check on our own fact: one row per shipment leg => 34 distinct legs
import pandas as pd
f = pd.read_csv("data/FactShipment.csv")
assert len(f) == f.ShipmentID.nunique() == 34, "grain broken!"
print("34 rows, 34 distinct ShipmentIDs -> grain holds: one row = one shipment leg")

34 rows, 34 distinct ShipmentIDs -> grain holds: one row = one shipment leg
